# yfinance Exploratory Data Analysis

In [72]:
# Imports
import yfinance as yf
import pandas as pd
import requests
from io import StringIO
from IPython.display import clear_output
import os

## Accessing Stock Data

In [ ]:
# Function to fetch stock data for a single ticker using specified start and end dates
def get_stock_data_using_dates(ticker, start_date, end_date, interval="1d"):
    try:
        stock_data = yf.Ticker(ticker).history(start=start_date, end=end_date, interval=interval)
        return stock_data
    except Exception as e:
        print(f"An error occurred while fetching stock data: {e}")
        return None

# Function to fetch stock data for a single ticker using a specified period
def get_stock_data_using_period(ticker, period, interval="1d"):
    try:
        stock_data = yf.Ticker(ticker).history(period=period, interval=interval)
        return stock_data
    except Exception as e:
        print(f"An error occurred while fetching stock data: {e}")
        return None

# Function to fetch the list of S&P 500 companies from Wikipedia
def get_snp_companies():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    table = pd.read_html(StringIO(response.text))[0]

    snp_df = table[["Symbol", "Security", "GICS Sector", "Date added"]].copy()
    snp_df["Symbol"] = snp_df["Symbol"].str.replace(".", "-", regex=False)

    return snp_df

# Example usage:
if __name__ == "__main__":

    # Fetch stock data using dates
    stock_data_dates = get_stock_data_using_dates("TSLA", "2023-01-01", "2023-12-31")
    print(stock_data_dates.head())

    # Fetch stock data using period
    stock_data_period = get_stock_data_using_period("TSLA", "max")

    # Fetch S&P 500 companies
    snp_companies = get_snp_companies()
    snp_companies.to_csv("../raw_data/snp_companies.csv", index=False)


                                 Open        High         Low       Close  \
Date                                                                        
2023-01-03 00:00:00-05:00  118.470001  118.800003  104.639999  108.099998   
2023-01-04 00:00:00-05:00  109.110001  114.589996  107.519997  113.639999   
2023-01-05 00:00:00-05:00  110.510002  111.750000  107.160004  110.339996   
2023-01-06 00:00:00-05:00  103.000000  114.389999  101.809998  113.059998   
2023-01-09 00:00:00-05:00  118.959999  123.519997  117.110001  119.769997   

                              Volume  Dividends  Stock Splits  
Date                                                           
2023-01-03 00:00:00-05:00  231402800        0.0           0.0  
2023-01-04 00:00:00-05:00  180389000        0.0           0.0  
2023-01-05 00:00:00-05:00  157986300        0.0           0.0  
2023-01-06 00:00:00-05:00  220911100        0.0           0.0  
2023-01-09 00:00:00-05:00  190284000        0.0           0.0  


## Preprocessing a Single Stock

Notes:
1. Open/Close is already adjusted for Stock Splits

In [67]:
tsla_history = get_stock_data_using_period("TSLA", "max")
tsla_history.to_csv("../raw_data/tsla_stock_data.csv")

# Add daily returns, moving averages, and volatility as technical features
def add_technical_features(df):
    df = df.copy()

    # Daily returns based off closing prices
    df["Return_1D"] = df["Close"].pct_change()

    # Moving averages based off closing prices
    df["MA_5"] = df["Close"].rolling(5).mean()
    df["MA_21"] = df["Close"].rolling(21).mean()
    df["MA_126"] = df["Close"].rolling(126).mean()
    df["MA_252"] = df["Close"].rolling(252).mean()

    # Volatility (standard deviation of returns) based off daily returns
    df["Volatility_21"] = df["Return_1D"].rolling(21).std()

    # Drop rows with NaN values resulting from rolling calculations
    df.dropna(inplace=True)

    # Remove Dividend and Stock Splits columns as they are not needed for modeling
    df.drop(columns=["Dividends", "Stock Splits"], inplace=True)

    return df

tsla_history_with_features = add_technical_features(tsla_history)
tsla_history_with_features.to_csv("../raw_data/tsla_stock_data_with_features.csv", index=True)

## Get all Preprocessed Data for Numerous/All Stocks in the SNP 500

In [81]:
# Option 1 : Get all tickers from raw/snp_companies.csv
# NOTE: This will result in 1GB of data being fetched and stored in processed_data
# If you want to fetch data for a smaller subset of tickers, you can modify the code below to only include a few tickers from the list.
snp_companies_df = pd.read_csv("../raw_data/snp_companies.csv")
#tickers = snp_companies_df["Symbol"].tolist()

# Option 2: Fetch data for a smaller subset of tickers
tickers = ["NVDA","AAPL","MSFT","AMZN","GOOGL","AVGO","META","GOOG","TSLA","BRK-B"]  # Example subset of tickers

# Fetch stock data using max period for all tickers
total_tickers = len(tickers)

for i, ticker in enumerate(tickers, start=1):
    stock_data_max_period = get_stock_data_using_period(ticker, period="max")
    preprocessed_data = add_technical_features(stock_data_max_period)
    preprocessed_data.to_csv(f"../processed_data/stock_data/{ticker}_stock_data.csv", index=True)

    stock_recommendations = yf.Ticker(ticker).get_recommendations()
    stock_recommendations.to_csv(f"../processed_data/recommendations/{ticker}_recommendations.csv", index=True)

    clear_output(wait=True)
    print(f"Fetched data for {ticker} ({i}/{total_tickers}). Progress: {i/total_tickers:.2%}")

Fetched data for BRK-B (10/10). Progress: 100.00%
